# **Core Analyses of Vitamin D Signatures**

## Section 1: Imports & Config.

In [ ]:
# Allow imports from src/vitd_utils
import sys
sys.path.append("../src")

# Core project utilities
from vitd_utils import config, idsymbols, coregenes, dose, gsea, plotting, stats, dataset

# Standard scientific libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from upsetplot import UpSet, from_contents

# Suppress specific warnings (e.g., from upsetplot)
import warnings
warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
    module="upsetplot"
)





# Display options
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 120)

# Variant/development notebook: keep plots visible, but do not write figure files.
# This prevents sensitivity analyses from overwriting manuscript outputs.
config.SAVE_FIGS = False
print("Variant figure saving disabled; plots will be displayed only.")


print("Results will be saved to:", config.RESULTS_DIR)


### 1.1. Load and validate

In [ ]:
EXP  = pd.read_parquet("../data/exports/expression_matrix_clean.parquet")
META = pd.read_csv("../data/exports/signature_metadata_clean.csv")

# Estandariza columnas (sig_id, cell_id, dose, analog)
META = dataset.standardize_meta(META)
print("Columns after standardize_meta:", META.columns.tolist())
print(META.filter(regex="dose", axis=1).head(2))  # ver 'dose' presente

# Alinea por sig_id
from vitd_utils import dataset as _ds
EXP, META = _ds.align_exp_meta(EXP, META)

## 2. Gene ID ↔ Symbol Mapping

In [ ]:
# Load gene metadata (example: geneinfo_beta.txt already loaded in previous steps)
gene_info = pd.read_csv("../data/raw_data/geneinfo_beta.txt", sep="\t")

# Build ID → symbol mapping
sym_map = idsymbols.build_symbol_map(gene_info)

# Quick check
print("Mapping size:", sym_map.shape[0])
print("Examples:\n", sym_map.head())

# Test safe fallback (ID not in map should return itself)
print("Test mapping:", idsymbols.map_symbols_or_ids(["100", "102", "999"], sym_map)[:5])

## 3. Consensus Core Genes (definition & scoring)

In [ ]:
# Sanity alignment: keep only signatures present in both matrices
common_sig = [c for c in EXP.columns if c in set(META["sig_id"])]
EXP = EXP[common_sig].copy()
META = META.loc[META["sig_id"].isin(common_sig)].copy()

# --- Build gene × cell effects (mean across signatures within each cell line)
cell_indexer = META.set_index("sig_id")["cell_id"]
effects_by_cell = EXP.T.groupby(cell_indexer).mean().T

print("effects_by_cell shape:", effects_by_cell.shape)
display(effects_by_cell.iloc[:5, :5])


## 4. Dose–Response Analysis

### 4.1 Prepare dose metadata

In [ ]:
assert "dose" in META.columns, "Expected 'dose' in META after standardization."

META["log_dose"] = np.log10(META["dose"])
META["dose_bin"] = META.groupby("cell_id")["dose"].transform(lambda d: dose.binarize_dose(d).values)

META[["sig_id", "cell_id", "dose", "log_dose", "dose_bin"]].head()

In [ ]:
# 1) Build gene × cell effects
effects_by_cell = dataset.effects_by_cell(EXP, META)  # genes × cell_id
print("effects_by_cell:", effects_by_cell.shape)

# 2) Consensus core sets (UP/DOWN)
cons = coregenes.build_consensus_core(
    effects_by_cell,
    top_n=config.N_TOP,
    min_votes=3,
    target_up=18,
    target_dn=6,
    min_non_na=10,
)
core_up_ids = cons["core_up"]
core_dn_ids = cons["core_dn"]
print(f"[core sets] UP={len(core_up_ids)} | DOWN={len(core_dn_ids)}")


# 3) Core score for every signature (columns of EXP)
core_scores = coregenes.core_score_for_matrix(
    effects=EXP,          # genes × sig_id
    core_up=core_up_ids,  # ID list (match EXP.index)
    core_dn=core_dn_ids,
    center=True,
)

# 4) Merge to META (standardized) by sig_id
if "core_score" in META.columns:
    META = META.drop(columns=["core_score"])
META = META.merge(core_scores.rename("core_score"),
                  left_on="sig_id", right_index=True, how="left")

# Sanity check
print("Has core_score?", "core_score" in META.columns, "| nulls:", META["core_score"].isna().sum())
display(META[["sig_id","cell_id","dose","core_score"]].head())


In [ ]:
set_up = set(core_up_ids)
set_dn = set(core_dn_ids)

In [ ]:
import pickle

with open("core_v3.pkl", "wb") as f:
    pickle.dump((set_up, set_dn), f)

In [ ]:
# ==========================================
# Audit: did 42 / 35 arise naturally?
# ==========================================

# Step 1: extreme genes per cell line
extremes = coregenes.top_bottom_by_column(
    effects_by_cell,
    top_n=config.N_TOP,   # 50
    min_non_na=10
)

# Step 2: vote counts
votes_up, votes_dn = coregenes.vote_counts(extremes)

# Step 3: genes passing the initial vote threshold only
up_ge2 = votes_up[votes_up >= config.VOTE_MIN].copy()
dn_ge2 = votes_dn[votes_dn >= config.VOTE_MIN].copy()

print(f"Genes with vote >= {config.VOTE_MIN}  | UP: {len(up_ge2)} | DOWN: {len(dn_ge2)}")

# Step 4: inspect vote count distributions
print("\nUP vote distribution:")
print(votes_up.value_counts().sort_index(ascending=False))

print("\nDOWN vote distribution:")
print(votes_dn.value_counts().sort_index(ascending=False))

# Step 5: tie-break scores among genes passing threshold
tie_up = effects_by_cell.reindex(up_ge2.index).clip(lower=0).mean(axis=1, skipna=True).sort_values(ascending=False)
tie_dn = (-effects_by_cell.reindex(dn_ge2.index).clip(upper=0)).mean(axis=1, skipna=True).sort_values(ascending=False)

audit_up = pd.DataFrame({
    "votes": up_ge2,
    "tie_score_up": tie_up.reindex(up_ge2.index)
}).sort_values(["votes", "tie_score_up"], ascending=[False, False])

audit_dn = pd.DataFrame({
    "votes": dn_ge2,
    "tie_score_dn": tie_dn.reindex(dn_ge2.index)
}).sort_values(["votes", "tie_score_dn"], ascending=[False, False])


### 4.2 Monotonicity test (Spearman ρ)

In [ ]:
# Per-cell monotonicity of core_score vs dose
mono_results = (
    META.groupby("cell_id")
        .apply(lambda sub: dose.dose_monotonicity(sub["dose"], sub["core_score"]))
        .apply(pd.Series)
        .reset_index()
)

print(mono_results)

### 4.3 Forest plot — Dose–response slopes (HC3)

In [ ]:
# Ensure prerequisites are available
assert "dose" in META.columns, "Dose missing — run Section 0 standardization first."
assert "core_score" in META.columns, "core_score missing — run Section 3 consensus core scoring first."

# 1) Fit OLS-HC3 models per cell line
models, labels = [], []
for cell, sub in META.groupby("cell_id", dropna=False):
    sub = sub[["dose", "core_score"]].dropna()
    # Require at least 3 samples and 2 unique dose values
    if sub["dose"].nunique(dropna=True) < 2 or len(sub) < 3:
        continue
    try:
        m = dose.ols_hc3(sub["dose"].values, sub["core_score"].values)
        models.append(m)
        labels.append(str(cell))
    except Exception:
        # Skip groups where regression fails numerically
        continue

# 2) Summarize and plot
if models:
    # Summarize slopes with robust CI and p-values
    summary_df = dose.summarize_forest(models, labels)
    display(summary_df.sort_values("coef"))

    # Optionally export summary table
    if getattr(config, "SAVE_TABLES", False):
        out_csv = config.RESULTS_DIR / "dose_response_forest_by_cell.csv"
        summary_df.to_csv(out_csv, index=False)
        print(f"[saved] {out_csv}")

    # Forest plot
    ax = plotting.forest_from_models(
        models, labels,
        title="Dose–response slope (core_score ~ log10 dose) by cell line",
        sort="coef"
    )
    plt.show()
else:
    print("[info] No groups had enough dose variation to fit OLS-HC3.")


In [ ]:
# Ensure prerequisites are available
assert "dose" in META.columns, "Dose missing — run Section 0 standardization first."
assert "core_score" in META.columns, "core_score missing — run Section 3 consensus core scoring first."

# 1) Fit OLS-HC3 models per cell line
models, labels = [], []
for cell, sub in META.groupby("cell_id", dropna=False):
    sub = sub[["dose", "core_score"]].dropna()
    # Require at least 3 samples and 2 unique dose values
    if sub["dose"].nunique(dropna=True) < 2 or len(sub) < 3:
        continue
    try:
        m = dose.ols_hc3(sub["dose"].values, sub["core_score"].values)
        models.append(m)
        labels.append(str(cell))
    except Exception:
        continue

# 2) Summarize and plot
if models:
    # Summarize slopes with robust CI and p-values
    summary_df = dose.summarize_forest(models, labels).sort_values("coef")
    display(summary_df)

    # Optionally export summary table
    if getattr(config, "SAVE_TABLES", False):
        out_csv = config.RESULTS_DIR / "dose_response_forest_by_cell.csv"
        summary_df.to_csv(out_csv, index=False)
        print(f"[saved] {out_csv}")

    # --- Forest plot (compact + no p-values on plot) ---
    # If your helper supports it, uncomment the next line and remove the manual cleanup below:
    # ax = plotting.forest_from_models(models, labels, title=..., sort="coef", show_pvalues=False)

    ax = plotting.forest_from_models(
        models, labels,
        title="Dose–response slope (core_score ~ log10 dose) by cell line",
        sort="coef"
    )

    # 1) Remove any text annotations that look like p-values (robust to unknown implementation)
    for t in list(ax.texts):
        s = t.get_text()
        if isinstance(s, str) and ("p=" in s or "p =" in s or "P=" in s or "P =" in s):
            t.remove()

    # 2) Make it less tall (shrink height). Adjust as you like.
    fig = ax.figure
    fig.set_size_inches(6.0, 2.8)  # <- lower height than before

    # Tighten layout so labels don't get clipped
    fig.tight_layout()

    plt.show()
else:
    print("[info] No groups had enough dose variation to fit OLS-HC3.")


---

## 5.1 Groupwise Spearman correlations (with FDR)

In [ ]:
# Ensure required columns exist
assert {"cell_id", "log_dose", "core_score"}.issubset(META.columns), \
    "Missing required columns. Run Sections 3 and 4.1 first."

# Prepare a minimal, clean table
df = META[["cell_id", "log_dose", "core_score"]].dropna().copy()

# Compute Spearman by group (prefer utils; fall back to a safe inline implementation if needed)
try:
    # Expected signature: stats.spearman_by_group(df, group_col, x_col, y_col)
    spearman_df = stats.spearman_by_group(df, group_col="cell_id",
                                          x_col="log_dose", y_col="core_score")
except Exception:
    # Safe fallback: pure pandas/scipy implementation
    from scipy.stats import spearmanr

    rows = []
    for cell, sub in df.groupby("cell_id", dropna=False):
        # Require at least 3 samples and 2 unique x values
        if sub["log_dose"].nunique() < 2 or len(sub) < 3:
            rows.append({"cell_id": str(cell), "rho": np.nan, "pvalue": np.nan, "n": len(sub)})
            continue
        r, p = spearmanr(sub["log_dose"], sub["core_score"], nan_policy="omit")
        rows.append({"cell_id": str(cell), "rho": r, "pvalue": p, "n": len(sub)})
    spearman_df = pd.DataFrame(rows)

# Add BH–FDR (prefer utils; fall back to statsmodels if needed)
try:
    # Expected signature: stats.add_fdr(df, p_col, new_col="fdr", method="bh")
    spearman_df = stats.add_fdr(spearman_df, p_col="pvalue", new_col="fdr", method="bh")
except Exception:
    from statsmodels.stats.multitest import multipletests
    p = spearman_df["pvalue"].values
    mask = ~np.isnan(p)
    fdr = np.full_like(p, np.nan, dtype=float)
    if mask.sum() > 0:
        _, q, _, _ = multipletests(p[mask], alpha=0.05, method="fdr_bh")
        fdr[mask] = q
    spearman_df["fdr"] = fdr

# Nicely formatted table
spearman_df = (
    spearman_df.rename(columns={"cell_id": "label", "rho": "spearman_rho"})
               .sort_values(["fdr", "spearman_rho"], ascending=[True, False])
               .reset_index(drop=True)
)

display(spearman_df)

# Optional: save results
if getattr(config, "SAVE_TABLES", False):
    out_csv = config.RESULTS_DIR / "spearman_by_cell_fdr.csv"
    spearman_df.to_csv(out_csv, index=False)
    print(f"[saved] {out_csv}")


## 5.2 Bootstrap CIs for Spearman’s ρ

In [ ]:
def _spearman_stat(xy_df: pd.DataFrame) -> float:
    # small helper that returns Spearman's rho
    from scipy.stats import spearmanr
    r, _ = spearmanr(xy_df["log_dose"], xy_df["core_score"], nan_policy="omit")
    return r

rows = []
for cell, sub in df.groupby("cell_id", dropna=False):
    sub = sub.copy()
    n = len(sub)
    if sub["log_dose"].nunique() < 2 or n < 3:
        rows.append({"label": str(cell), "rho": np.nan, "ci_low": np.nan, "ci_high": np.nan, "n": n})
        continue
    # Prefer utils if available; otherwise fallback to a simple bootstrap
    try:
        # Expected signature: stats.bootstrap_ci(data, stat_fn, n_boot=2000, ci=0.95, seed=0)
        ci_low, ci_high, point = stats.bootstrap_ci(sub, _spearman_stat, n_boot=4000, ci=0.95, seed=0)
        rows.append({"label": str(cell), "rho": point, "ci_low": ci_low, "ci_high": ci_high, "n": n})
    except Exception:
        # Basic bootstrap fallback
        rng = np.random.default_rng(0)
        boots = []
        for _ in range(4000):
            idx = rng.integers(0, n, n)  # sample with replacement
            boots.append(_spearman_stat(sub.iloc[idx]))
        boots = np.array(boots)
        ci_low, ci_high = np.nanpercentile(boots, [2.5, 97.5])
        rows.append({"label": str(cell), "rho": _spearman_stat(sub), "ci_low": ci_low, "ci_high": ci_high, "n": n})

boot_df = pd.DataFrame(rows).sort_values("rho", ascending=False).reset_index(drop=True)
display(boot_df)

# Optional: save table
if getattr(config, "SAVE_TABLES", False):
    out_csv = config.RESULTS_DIR / "spearman_by_cell_bootstrap_ci.csv"
    boot_df.to_csv(out_csv, index=False)
    print(f"[saved] {out_csv}")

# Point-interval plot
plt.figure(figsize=(6, 3.6))
y = np.arange(len(boot_df))[::-1]
plt.hlines(y, boot_df["ci_low"], boot_df["ci_high"])
plt.plot(boot_df["rho"], y, "o")
plt.axvline(0, linestyle="--", linewidth=1)
plt.yticks(y, boot_df["label"])
plt.xlabel("Spearman ρ (with 95% CI)")
plt.title("Dose–response monotonicity by cell line")
plt.tight_layout()
plt.show()

---


## 5.3 Quick OLS slopes (polyfit).


In [ ]:
# Ensure required columns are available
assert {"cell_id", "log_dose", "core_score"}.issubset(META.columns)

rows = []
for cell, sub in META.groupby("cell_id", dropna=False):
    slope = stats.fit_slope_ols(sub, x="log_dose", y="core_score")
    rows.append({"label": str(cell), "slope": slope, "n": len(sub)})

slopes_df = pd.DataFrame(rows).sort_values("slope", ascending=False).reset_index(drop=True)
display(slopes_df)

# Optional: save results
if getattr(config, "SAVE_TABLES", False):
    out_csv = config.RESULTS_DIR / "quick_slopes_by_cell.csv"
    slopes_df.to_csv(out_csv, index=False)
    print(f"[saved] {out_csv}")

# Simple bar plot of slopes
plt.figure(figsize=(6, 3.6))
plt.barh(slopes_df["label"], slopes_df["slope"], color="steelblue")
plt.axvline(0, linestyle="--", linewidth=1, color="black")
plt.xlabel("OLS slope (Δ core_score per Δ log10 dose)")
plt.title("Quick dose–response slopes by cell line")
plt.tight_layout()
plt.show()


---


## 5.4 Bootstrap CIs for OLS slopes

In [ ]:
rows = []
for cell, sub in META.groupby("cell_id", dropna=False):
    sub = sub[["log_dose", "core_score"]].dropna()
    n = len(sub)
    if sub["log_dose"].nunique() < 2 or n < 3:
        rows.append({"label": str(cell), "slope": np.nan, "ci_low": np.nan, "ci_high": np.nan, "n": n})
        continue
    # Bootstrap CI using the utility if available
    try:
        lo, hi = stats.bootstrap_ci(stats.fit_slope_ols(sub), B=4000, alpha=0.05)
        slope_val = stats.fit_slope_ols(sub)
    except Exception:
        # Manual fallback
        rng = np.random.default_rng(0)
        boots = []
        x, y = sub["log_dose"].values, sub["core_score"].values
        for _ in range(4000):
            idx = rng.integers(0, n, n)
            slope = stats.fit_slope_ols(pd.DataFrame({"log_dose": x[idx], "core_score": y[idx]}))
            boots.append(slope)
        boots = np.array([b for b in boots if not np.isnan(b)])
        slope_val = stats.fit_slope_ols(sub)
        lo, hi = np.nanpercentile(boots, [2.5, 97.5])
    rows.append({"label": str(cell), "slope": slope_val, "ci_low": lo, "ci_high": hi, "n": n})

boot_slopes_df = pd.DataFrame(rows).sort_values("slope", ascending=False).reset_index(drop=True)
display(boot_slopes_df)

# Optional: save
if getattr(config, "SAVE_TABLES", False):
    out_csv = config.RESULTS_DIR / "quick_slopes_bootstrap_ci.csv"
    boot_slopes_df.to_csv(out_csv, index=False)
    print(f"[saved] {out_csv}")

# Plot
plt.figure(figsize=(6, 3.6))
y = np.arange(len(boot_slopes_df))[::-1]
plt.hlines(y, boot_slopes_df["ci_low"], boot_slopes_df["ci_high"])
plt.plot(boot_slopes_df["slope"], y, "o")
plt.axvline(0, linestyle="--", linewidth=1)
plt.yticks(y, boot_slopes_df["label"])
plt.xlabel("OLS slope (Δ core_score per Δ log10 dose) with 95% CI")
plt.title("Bootstrap CIs for quick dose–response slopes")
plt.tight_layout()
plt.show()


---

## 6.1 Preranked gene lists for enrichment

In [ ]:
# Use per-cell effects (from Section 3)
# effects_by_cell: genes × cell_id matrix of mean z-scores
assert "effects_by_cell" in globals(), "Run Section 3 consensus first."

# Map IDs → symbols (safe fallback to IDs if missing)
sym_map = idsymbols.build_symbol_map(pd.read_csv("../data/raw_data/geneinfo_beta.txt", sep="\t"))

# Create preranked lists per cell line
ranked_lists = {}
for cell in effects_by_cell.columns:
    vec = effects_by_cell[cell].dropna()
    df = gsea.make_preranked(vec, sym_map=sym_map)
    ranked_lists[cell] = df
    print(f"{cell}: {df.shape[0]} genes ranked")

# Example preview
ranked_lists["MCF7"].head(10)

# Optionally save preranked lists
if getattr(config, "SAVE_TABLES", False):
    out_dir = config.RESULTS_DIR / "preranked_lists"
    out_dir.mkdir(exist_ok=True)
    for cell, df in ranked_lists.items():
        df.to_csv(out_dir / f"preranked_{cell}.rnk", sep="\t", index=False, header=False)
    print(f"[saved] {len(ranked_lists)} preranked .rnk files in {out_dir}")

## 6.2 GSEA (resumable, per cell line)

In [ ]:
# Make preranked lists discoverable by the GSEA runner
# (the runner looks for gsea.gsea_preranked_by_cell / _by_analog in module globals)
gsea.gsea_preranked_by_cell = ranked_lists  # {cell: DataFrame('gene','score')}

# Define libraries (paths or in-memory dicts). Defaults come from config.
libraries = config.GSEA_LIBRARIES  # e.g., Hallmarks, Reactome GMTs under ROOT_DIR/libs

# Run resumable enrichment (uses checkpoints in config.CHECKPOINT_DIR)
enr_results = gsea.run_enrichment_axis_resumable(
    axis="cell",
    libraries=libraries,
    target_perms=config.PERM_N,
    chunk_perms=config.CHUNK_PERMS,
    min_size=15,
    max_size=500,
    random_state=config.SEED,
)

# Preview: top rows for each library × first cell
for lib_name, by_group in enr_results.items():
    # pick first non-empty group
    first_nonempty = next((g for g, df in by_group.items() if df is not None and not df.empty), None)
    if first_nonempty is None:
        print(f"[warn] No results for {lib_name} — check library path or preranked input.")
        continue
    print(f"\n[{lib_name}] example group: {first_nonempty}")
    display(by_group[first_nonempty].head())


---

## 6.3 Dot-plot of top enriched pathways


In [ ]:
for lib_name, by_group in enr_results.items():
    gsea.dotplot_top(
        enr_results=by_group,
        lib_name=lib_name,
        axis="cell",
        top_n=10,
        fdr_cutoff=0.05,
        groups=["A549","HA1E","MCF7","PC3","U2OS"],
        wrap_width=80,
        vmax_cap=2.5,
        fig_width=9.0,
        left_margin=0.5,
        bottom_pad=0.33,
        point_sizes=(26,140),
        )

In [ ]:
# Count how many cell lines each term is significant in (FDR < 0.05)
consensus = []
for lib_name, by_group in enr_results.items():
    rows = []
    for cell, df in by_group.items():
        if df is None or df.empty:
            continue
        sig = df[df["fdr_bh"] < 0.05].copy()
        sig["cell_id"] = cell
        rows.append(sig[["term","ES","fdr_bh","cell_id"]])
    if not rows:
        continue
    cat = pd.concat(rows, ignore_index=True)
    tally = (cat.groupby("term")
                .agg(n_cells=("cell_id","nunique"),
                     mean_ES=("ES","mean"))
                .sort_values(["n_cells","mean_ES"], ascending=[False, False])
                .reset_index())
    tally["library"] = lib_name
    consensus.append(tally)

consensus_df = pd.concat(consensus, ignore_index=True) if consensus else pd.DataFrame()
print(consensus_df.head(20))


---

In [ ]:
# 1) Filtrar Hallmarks y elegir top N
topN = 10
dfC = (
    consensus_df
    .query("library == 'Hallmarks'")
    .sort_values(["n_cells", "mean_ES"], ascending=[False, False])
    .head(topN)
    .copy()
)

# 2) Orden para plot
dfC["term"] = pd.Categorical(dfC["term"], categories=dfC["term"][::-1], ordered=True)

plt.figure(figsize=(6, 3.5))
ax = sns.barplot(data=dfC, x="mean_ES", y="term")

ax.set_xlabel("Mean enrichment score across cell lines")
ax.set_ylabel("")

# 3) Etiqueta con n_cells al final de cada barra
for i, row in dfC.reset_index(drop=True).iterrows():
    ax.text(row["mean_ES"] + 0.01, i, f"n={int(row['n_cells'])}", va="bottom", fontsize=8)

plt.title("Consensus Hallmark enrichment across cell lines", fontsize=10, x=0.02 )
plt.tight_layout()

plt.show()


In [ ]:
dfC.head()

## 7. Visualization of directed results

In [ ]:
core_scores_df = META[["sig_id", "cell_id", "log_dose", "core_score"]].dropna()

forest_df = stats.summarize_slopes_ols_hc3(core_scores_df)
ax = plotting.forest_slopes(forest_df, title="Dose–response slopes by cell line", sort="coef")


---

###  7.2 Box + strip plots — core scores by dose and cell line

In [ ]:
ax = plotting.box_strip(
    META, x="dose_bin", y="core_score", hue="cell_id",
    title="Core scores across doses and cell lines"
)
plt.show()

---

### 7.3 Dot plots — top enriched pathways

In [ ]:
for lib_name, by_group in enr_results.items():
    ax = plotting.dotplot_top(
        by_group,
        top_n=10,
        fdr_cutoff=0.05,
        groups=["A549","HA1E","MCF7","PC3","U2OS"],
        wrap_width=80,
        vmax_cap=2.5,
        fig_width=14,
        point_sizes=(26, 140),
        title=f"Enrichment summary — {lib_name} by cell"
    )
    plt.show()


---

## Heatmap gene-level effects across cell lines

In [ ]:
effects_by_cell.head()

In [ ]:
top_genes_named = pd.read_csv("../data/exports/functional_context/top_genes_named.csv")
print(top_genes_named.head())

In [ ]:
# --- Align top genes (Panel A) with effects_by_cell (Panel C) ---

# genes del Panel A (gene_id)
genes_A = top_genes_named["gene_id"]

# quedarnos solo con los que existen en effects_by_cell
genes_A_present = genes_A[genes_A.isin(effects_by_cell.index)]

print(f"Genes in Panel A: {len(genes_A)}")
print(f"Genes present in effects_by_cell: {len(genes_A_present)}")


In [ ]:
# subset del effects_by_cell
hm_df = effects_by_cell.loc[genes_A_present].copy()

# mapping gene_id → gene_symbol
gene_map = (
    top_genes_named
    .set_index("gene_id")["gene_symbol"]
)

hm_df.index = gene_map.loc[genes_A_present].values

print(hm_df.shape)
print(hm_df.head())


In [ ]:
# order rows by mean_abs_z from Panel A (ranking)
order = (
    top_genes_named
    .set_index("gene_symbol")
    .loc[hm_df.index]["0"]
    .sort_values(ascending=False)
    .index
)

hm_df = hm_df.loc[order]

In [ ]:
# Plot heatmap of gene-level effects across cell lines
plt.figure(figsize=(4, 3))

sns.heatmap(
    hm_df,
    cmap="vlag",
    center=0,
    linewidths=0.3,
    linecolor="white",
    cbar_kws={"label": "Effect size"},
)

plt.xlabel("")
plt.ylabel("Gene symbol")
plt.title("Gene-level effects across cell lines", fontsize=11)

plt.tight_layout()


plt.show()

### Build gene × cell effects (mean across signatures within each cell line)

In [ ]:
# 0) Sanity checks
required_meta_cols = {"sig_id", "cell_id"}
missing = required_meta_cols - set(META.columns)
if missing:
    raise ValueError(f"META is missing required columns: {missing}")

# Ensure META has only sig_ids present in EXP columns (and same order doesn't matter)
meta = META.loc[META["sig_id"].isin(EXP.columns)].copy()
if meta.empty:
    raise ValueError("After filtering, META has 0 rows matching EXP columns. Check sig_id alignment.")

# 1) Build mapping: cell_id -> list of sig_ids
cell_to_sigs = (
    meta.groupby("cell_id")["sig_id"]
    .apply(list)
    .to_dict()
)

# 2) Aggregate per cell line
rows = []
for cell_id, sig_ids in cell_to_sigs.items():
    # restrict to signatures that actually exist in EXP (defensive)
    sig_ids = [s for s in sig_ids if s in EXP.columns]
    if len(sig_ids) == 0:
        continue

    X = EXP[sig_ids]  # genes × signatures (z-scores)
    z_mean = X.mean(axis=1)
    z_std = X.std(axis=1, ddof=1)  # sample std
    n_sig = len(sig_ids)

    tmp = pd.DataFrame({
        "gene": X.index,
        "cell_id": cell_id,
        "z_mean": z_mean.values,
        "z_std": z_std.values,
        "n_signatures": n_sig,
        "abs_z_mean": np.abs(z_mean.values),
    })
    rows.append(tmp)

gene_cell_effects = pd.concat(rows, ignore_index=True)

# 3) Quick sanity outputs
print("gene_cell_effects shape:", gene_cell_effects.shape)
print("cell lines:", gene_cell_effects["cell_id"].unique())
print(gene_cell_effects.head())


In [ ]:
# --- Define modulated genes (data-driven threshold per cell line)

# Percentile threshold
PCTL = 0.90

# 1) Compute per–cell line threshold on |z_mean|
thr = (
    gene_cell_effects
    .groupby("cell_id")["abs_z_mean"]
    .quantile(PCTL)
    .rename("thr_p90")
    .reset_index()
)

# 2) Merge thresholds back
gene_cell_effects = gene_cell_effects.merge(
    thr, on="cell_id", how="left"
)

# 3) Define modulation flag
gene_cell_effects["is_modulated"] = (
    gene_cell_effects["abs_z_mean"] >= gene_cell_effects["thr_p90"]
)

# 4) Sanity checks
summary = (
    gene_cell_effects
    .groupby("cell_id")["is_modulated"]
    .agg(n_modulated="sum", total="count")
    .assign(frac=lambda d: d["n_modulated"] / d["total"])
)

print(summary)


In [ ]:
# --- Build gene sets per cell line for UpSet plot

# Dictionary: cell_id -> set of modulated genes
gene_sets = {
    cell: set(
        gene_cell_effects
        .loc[
            (gene_cell_effects["cell_id"] == cell) &
            (gene_cell_effects["is_modulated"]),
            "gene"
        ]
        .astype(str)
    )
    for cell in gene_cell_effects["cell_id"].unique()
}

# Sanity check
for cell, genes in gene_sets.items():
    print(f"{cell}: {len(genes)} genes")


In [ ]:
# Convert sets to UpSet format
upset_data = from_contents(gene_sets)

# Plot
plt.figure(figsize=(8, 4))
up = UpSet(
    upset_data,
    show_counts=True,
    show_percentages=False,
    sort_by="cardinality",
    min_subset_size=10
)
up.plot()

plt.suptitle("Shared and cell line–specific vitamin D–responsive genes", y=0.95, fontsize=12)


plt.show()


In [ ]:
# --- Get genes shared across ALL cell lines (the 5-way intersection)
all_cells = list(gene_sets.keys())
shared_all = set.intersection(*(gene_sets[c] for c in all_cells))

print("Cell lines:", all_cells)
print("Shared genes across all cell lines:", len(shared_all))
print("Shared gene IDs (sorted):")
print(sorted(shared_all)[:50])  # preview


In [ ]:
# shared_all is a set of strings with Entrez IDs
shared_ids = set(map(int, shared_all))  # convert to int

shared_genes_df = (
    gene_info
    .loc[gene_info["gene_id"].isin(shared_ids)]
    .copy()
    .sort_values("gene_symbol")
)

print(shared_genes_df[["gene_id", "gene_symbol", "gene_title"]])
print("n rows:", shared_genes_df.shape[0])

In [ ]:
print(gene_info.columns)
print(gene_info.head())


In [ ]:
import gseapy as gp

lib = gp.get_library(name="MSigDB_Hallmark_2020", organism="Human")

In [ ]:
# Genes compartidos
shared_symbols = set(shared_genes_df["gene_symbol"])

rows = []
for gene in sorted(shared_symbols):
    memberships = [
        hallmark for hallmark, genes in lib.items()
        if gene in genes
    ]
    rows.append({
        "gene_symbol": gene,
        "hallmark_membership": ", ".join(sorted(memberships)) if memberships else "Not assigned"
    })

shared_hallmark_table = pd.DataFrame(rows)
print(shared_hallmark_table)


In [ ]:
hallmark_counts = (
    shared_hallmark_table
    .assign(hallmark=shared_hallmark_table["hallmark_membership"].str.split(", "))
    .explode("hallmark")
    .query("hallmark != 'Not assigned'")
    .groupby("hallmark")["gene_symbol"]
    .nunique()
    .sort_values(ascending=False)
)

hallmark_counts


In [ ]:
from docx import Document

# Create Word document
doc = Document()
doc.add_heading("Table 1. Hallmark membership of genes shared across all cell lines", level=2)

table = doc.add_table(
    rows=1,
    cols=2
)
table.style = "Table Grid"

# Header
hdr_cells = table.rows[0].cells
hdr_cells[0].text = "Gene"
hdr_cells[1].text = "Hallmark membership"

# Rows
for _, row in shared_hallmark_table.iterrows():
    cells = table.add_row().cells
    cells[0].text = row["gene_symbol"]
    cells[1].text = row["hallmark_membership"]

# Save
out_path = config.RESULTS_DIR / "Table_1_shared_gene_hallmark_membership.docx"
doc.save(out_path)

out_path


### Effect size comparison (shared vs cell line–specific)

In [ ]:
# 1) Count in how many cell lines each gene is modulated
mod_counts = (
    gene_cell_effects
    .loc[gene_cell_effects["is_modulated"]]
    .groupby("gene")["cell_id"]
    .nunique()
    .rename("n_cells")
    .reset_index()
)

# 2) Merge back
dfB = gene_cell_effects.merge(mod_counts, on="gene", how="left")

# 3) Define groups
dfB = dfB.loc[dfB["is_modulated"]].copy()

dfB["group"] = np.where(
    dfB["n_cells"] == 5,
    "Shared (5 cell lines)",
    np.where(dfB["n_cells"] == 1, "Cell line–specific", "Other")
)

# Keep only the two groups of interest
dfB = dfB.loc[dfB["group"].isin(["Shared (5 cell lines)", "Cell line–specific"])]

print(dfB["group"].value_counts())

In [ ]:
# 4) Plot box + strip of |z_mean| by group
plt.figure(figsize=(4, 3))

ax = sns.boxplot(
    data=dfB,
    x="group",
    y="abs_z_mean",
    showfliers=False
)

sns.stripplot(
    data=dfB,
    x="group",
    y="abs_z_mean",
    color="black",
    size=3,
    alpha=0.4,
    jitter=True
)

ax.set_xlabel("")
ax.set_ylabel("Mean absolute z-score")


plt.suptitle(    "Effect size distribution of shared versus\n"
                 "cell line–specific vitamin D–responsive genes", y=0.95, fontsize=12)
plt.tight_layout()


plt.show()


### Panel C — Hallmark enrichment patterns across cell lines (heatmap)

Heatmap of Hallmark pathway enrichment across the five cell lines to highlight shared versus context-dependent functional programs.



In [ ]:
# ------------------------------------------------------------
# Panel C — Hallmark enrichment patterns across cell lines
# Heatmap highlighting shared versus context-dependent pathways
# ------------------------------------------------------------

# -----------------------------
# 0) Sanity checks
# -----------------------------
# Ensure that GSEA results per cell line are available
assert "enr_results" in globals(), (
    "`enr_results` not found. Please run the GSEA per cell line section first."
)

# Extract Hallmark results
hall_by_cell = enr_results.get("Hallmarks", None)
assert hall_by_cell is not None, (
    "`enr_results['Hallmarks']` not found. Check available keys in `enr_results`."
)

# -----------------------------
# 1) Concatenate Hallmark GSEA results across cell lines
# -----------------------------
# Build a long-format DataFrame with one row per (pathway, cell line)
rows = []
for cell_id, df in hall_by_cell.items():
    if df is None or len(df) == 0:
        continue

    tmp = df.copy()

    # Ensure pathway names are stored in a column
    if "term" not in tmp.columns:
        tmp = tmp.reset_index().rename(columns={"index": "term"})

    tmp["cell_id"] = cell_id
    rows.append(tmp)

hall_long = pd.concat(rows, ignore_index=True)

# -----------------------------
# 2) Select the enrichment score metric
# -----------------------------
# Prefer NES if available; otherwise fall back to ES
if "NES" in hall_long.columns:
    value_col = "NES"
elif "nes" in hall_long.columns:
    value_col = "nes"
elif "ES" in hall_long.columns:
    value_col = "ES"
else:
    raise ValueError(
        "No enrichment score column found (NES/nes/ES). "
        f"Available columns: {list(hall_long.columns)}"
    )

# Detect an FDR/q-value column if present (not used for coloring here)
fdr_col = None
for c in ["fdr_bh", "FDR q-val", "fdr", "padj", "qval"]:
    if c in hall_long.columns:
        fdr_col = c
        break

# -----------------------------
# 3) Pathway selection (non–cherry-picked, reproducible criterion)
# -----------------------------
# Select top pathways based on:
#   (i) presence across cell lines (>= 4 cell lines)
#  (ii) highest average enrichment score across cell lines
topN = 10

presence = (
    hall_long
    .groupby("term")["cell_id"]
    .nunique()
    .rename("n_cells_present")
)

mean_score = (
    hall_long
    .groupby("term")[value_col]
    .mean()
    .rename("mean_score_across_cells")
)

ranking = (
    pd.concat([presence, mean_score], axis=1)
    .reset_index()
    .query("n_cells_present >= 4")
    .sort_values(
        ["n_cells_present", "mean_score_across_cells"],
        ascending=[False, False]
    )
)

top_terms = ranking.head(topN)["term"].tolist()

hall_long = hall_long[hall_long["term"].isin(top_terms)].copy()

# -----------------------------
# 4) Pivot to pathway × cell line matrix
# -----------------------------
cell_order = ["A549", "HA1E", "MCF7", "PC3", "U2OS"]

heatmap_df = (
    hall_long
    .pivot_table(
        index="term",
        columns="cell_id",
        values=value_col,
        aggfunc="mean"
    )
)

# Preserve ranking order for pathways
heatmap_df = heatmap_df.loc[top_terms]

# Preserve predefined cell line order where available
heatmap_df = heatmap_df[[c for c in cell_order if c in heatmap_df.columns]]

# Reverse row order for visual emphasis (top pathways at the top)
heatmap_plot = heatmap_df.iloc[::-1]

# -----------------------------
# 5) Plot heatmap (publication-ready)
# -----------------------------
plt.figure(figsize=(7.5, 3.6))

ax = sns.heatmap(
    heatmap_plot,
    cmap="viridis",
    vmin=0,
    vmax=heatmap_plot.values.max(),    
    linewidths=0.5,
    linecolor="white",
    cbar_kws={"label": value_col}
)

ax.set_xlabel("")
ax.set_ylabel("")
ax.set_title(
    "Hallmark enrichment patterns across cell lines for vitamin D–related perturbations", x=0.01, fontsize=14, y=1.02
)

plt.tight_layout()


plt.show()

print(f"[Panel C] Heatmap generated using enrichment score: {value_col}")
if fdr_col:
    print(f"[Info] FDR/q-value column detected (not used for color scale): {fdr_col}")


In [ ]:
print(gene_cell_effects[gene_cell_effects['gene']=='7421'])

In [ ]:
print(gene_cell_effects[
    gene_cell_effects["gene"].isin([7421, 1591, 1594, 6256])
][["gene", "cell_id", "z_mean", "is_modulated"]])


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# --- Panel D: VDR-axis transcript-level sanity check (mini heatmap)

# Entrez IDs for key genes in the vitamin D axis
genes_of_interest = {
    "VDR": 7421,
    "CYP24A1": 1591,
    "CYP27B1": 1594,
    "RXRA": 6256,
}

cell_order = ["A549", "HA1E", "MCF7", "PC3", "U2OS"]

# Build a small matrix: cell line × gene (z_mean)
dfD_long = (
    gene_cell_effects
    .loc[gene_cell_effects["gene"].isin(list(genes_of_interest.values())),
         ["gene", "cell_id", "z_mean"]]
    .copy()
)

# Map Entrez -> symbol for display
id_to_symbol = {v: k for k, v in genes_of_interest.items()}
dfD_long["gene_symbol"] = dfD_long["gene"].map(id_to_symbol)

heatD = (
    dfD_long
    .pivot(index="cell_id", columns="gene_symbol", values="z_mean")
    .reindex(cell_order)
)

# Plot
plt.figure(figsize=(3.4, 4))
ax = sns.heatmap(
    heatD,
    cmap="viridis",
    linewidths=0.5,
    linecolor="white",
    cbar_kws={"label": "Mean z-score (gene-level)"},
)

ax.set_xlabel("")
ax.set_ylabel("")
ax.set_title("VDR-axis gene expression changes\n(across cell lines)")

# Rotate x-axis labels (genes) to 45 degrees
plt.xticks(rotation=45, ha="right")

# Keep y-axis labels (cell lines) horizontal
plt.yticks(rotation=0)


plt.tight_layout()

plt.show()
